# Candidate Generation: Popularity Baseline

This notebook fits a non-personalised popularity model, retrieves time-eligible Top-M candidates, writes the standard candidate format, and evaluates it with `cg_evaluation.py`.

Standard candidate format:

```text
impression_uid, cutoff, news_code
```

Each impression has exactly `CANDIDATE_CAP` rows, with `cutoff = 1..CANDIDATE_CAP`. `cutoff` defines nested Top-M membership; it is not a ranking metric.

Protocol:

- Fit popularity on `train` for validation and on `train + validation` for test.
- Order articles by prior click count, breaking ties by ascending `news_code`.
- Restrict candidates to `available_from <= timestamp`.
- Expand the timestamp-level popularity order into impression-level candidates.
- Evaluate Recall@M at M = 50, 100, 200 and 500.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import os
import sys

import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

CODE_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_all.py').is_file()
)
ROOT = Path(os.environ.get('NEWSREC_ROOT', CODE_ROOT / 'runs/mindsmall-epoch')).expanduser().resolve()
sys.path.insert(0, str(CODE_ROOT / '5_Models' / '5_01_Candidate_Generation'))
from cg_evaluation import CandidateEvaluator

PROCESSED = ROOT / '1_Data_Preparation' / 'processed'
REPRESENTATIONS = ROOT / '3_Representations' / 'artifacts'
OUTPUT_DIR = ROOT / '5_Models' / '5_01_Candidate_Generation' / 'artifacts' / 'popularity'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SPLIT = 'validation'
FIT_SPLIT = 'train' if TARGET_SPLIT == 'validation' else 'train+validation'
M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)

if TARGET_SPLIT not in {'validation', 'test'}:
    raise ValueError('TARGET_SPLIT must be validation or test')

print('Repository:', ROOT)
print('Target split:', TARGET_SPLIT)
print('Fit split:', FIT_SPLIT)
print('M values:', M_VALUES)
print('Candidate cap:', CANDIDATE_CAP)

Repository: c:\Users\Yong Rui\BT4222-Group-6-
Target split: validation
Fit split: train
M values: (50, 100, 200, 500)
Candidate cap: 500


## 1. Fit the popularity table

The catalogue already stores split-level exposure and click counts, so there is no need to rescan the 70M training candidate rows. Those summaries are retrospective for their own split, which is safe because the entire fit split precedes the target split.

In [ ]:
catalogue = pl.read_parquet(
    PROCESSED / 'news_catalogue.parquet',
    columns=[
        'news_code',
        'exposures_train',
        'clicks_train',
        'exposures_validation',
        'clicks_validation',
    ],
)
availability = pl.read_parquet(
    REPRESENTATIONS / 'article_availability.parquet',
    columns=['news_code', 'available_from'],
)

prior_exposures = pl.col('exposures_train')
prior_clicks = pl.col('clicks_train')
if FIT_SPLIT == 'train+validation':
    prior_exposures = prior_exposures + pl.col('exposures_validation')
    prior_clicks = prior_clicks + pl.col('clicks_validation')

popularity = (
    catalogue.select(
        'news_code',
        prior_exposures.alias('prior_exposures'),
        prior_clicks.alias('prior_clicks'),
    )
    .with_columns(pl.col('prior_exposures').cast(pl.Int64).alias('popularity_score'))
    .join(availability, on='news_code', how='left')
    .sort(['popularity_score', 'news_code'], descending=[True, False])
)

if popularity.height != catalogue.height:
    raise ValueError('Popularity table lost catalogue rows')
if popularity['available_from'].null_count():
    raise ValueError('Missing available_from for some catalogue articles')
if popularity['popularity_score'].null_count():
    raise ValueError('Missing popularity scores')

model_path = OUTPUT_DIR / f"model_{FIT_SPLIT.replace('+', '_')}.parquet"
popularity.write_parquet(model_path)

print(f'Catalogue articles: {popularity.height:,}')
print(f'Prior clicks used for fitting: {popularity["popularity_score"].sum():,}')
print(f'Articles with at least one prior click: {popularity.filter(pl.col("popularity_score") > 0).height:,}')
print('Model saved to:', model_path)
popularity.head(10)

Catalogue articles: 104,151
Prior clicks used for fitting: 70,314,387
Articles with at least one prior click: 24,641
Model saved to: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\popularity\model_train.parquet


news_code,prior_exposures,prior_clicks,popularity_score,available_from
i32,i64,i64,i64,datetime[μs]
95525,331542,12366,331542,2019-11-09 00:00:17
53474,275636,26612,275636,2019-11-12 16:26:24
13761,260492,15759,260492,2019-11-09 00:46:22
98178,260330,60638,260330,2019-11-10 16:24:02
20990,241468,23085,241468,2019-11-11 16:00:34
10285,231349,8556,231349,2019-11-12 16:00:10
100073,228718,15855,228718,2019-11-11 16:23:03
61065,221509,15855,221509,2019-11-11 16:04:38
32154,220303,47078,220303,2019-11-09 17:48:11


## 2. Build the timestamp-level popularity order

Popularity is identical for every user, so only eligibility changes between requests. Build the ordered eligible list once per timestamp, then expand it to the standard impression-level format.

In [ ]:
impressions = pl.read_parquet(
    PROCESSED / f'impressions_{TARGET_SPLIT}.parquet',
    columns=['impression_uid', 'user_code', 'timestamp'],
)

timestamps = impressions['timestamp'].unique().sort()
ts = timestamps.to_numpy()
news_codes = popularity['news_code'].to_numpy()
available_from = popularity['available_from'].to_numpy()

n_timestamps = len(ts)
candidate_news = np.empty((n_timestamps, CANDIDATE_CAP), dtype=np.int32)

for i, timestamp in enumerate(ts):
    eligible_rows = np.flatnonzero(available_from <= timestamp)
    if eligible_rows.size < CANDIDATE_CAP:
        raise ValueError(f'Only {eligible_rows.size} eligible articles at {timestamp}')
    candidate_news[i] = news_codes[eligible_rows[:CANDIDATE_CAP]]

print(f'Target impressions: {impressions.height:,}')
print(f'Unique timestamps: {n_timestamps:,}')
print(f'Candidates per timestamp: {CANDIDATE_CAP}')
print('Top 5 articles at first timestamp:', candidate_news[0, :5].tolist())

Target impressions: 334,913
Unique timestamps: 54,728
Candidates per timestamp: 500
Top 5 articles at first timestamp: [95525, 53474, 13761, 98178, 20990]


## 3. Write the standard impression-level candidates

Each impression receives `CANDIDATE_CAP` rows. The write is batched so the full expansion does not need to sit in memory at once.

In [ ]:
recommendations_path = (
    OUTPUT_DIR / f'recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet'
)
timestamp_to_index = {timestamp: i for i, timestamp in enumerate(ts)}
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
candidate_schema = pa.schema(
    [
        ('impression_uid', pa.int64()),
        ('cutoff', pa.int16()),
        ('news_code', pa.int32()),
    ]
)

writer = pq.ParquetWriter(recommendations_path, candidate_schema, compression='zstd')
try:
    for batch in impressions.select('impression_uid', 'timestamp').iter_slices(
        n_rows=25_000
    ):
        batch_timestamps = batch['timestamp'].to_numpy()
        indices = np.fromiter(
            (timestamp_to_index[t] for t in batch_timestamps),
            dtype=np.int64,
            count=batch.height,
        )
        expanded = pl.DataFrame(
            {
                'impression_uid': np.repeat(
                    batch['impression_uid'].to_numpy(), CANDIDATE_CAP
                ),
                'cutoff': np.tile(cutoff_values, batch.height),
                'news_code': candidate_news[indices].reshape(-1),
            }
        )
        writer.write_table(expanded.to_arrow())
finally:
    writer.close()

expected_rows = impressions.height * CANDIDATE_CAP
print(f'Expected candidate rows: {expected_rows:,}')
print('Candidates saved to:', recommendations_path)

Expected candidate rows: 167,456,500
Candidates saved to: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\popularity\recommendations_validation_top500.parquet


## 4. Check the saved candidate artifact

In [ ]:
saved_rows = pl.scan_parquet(recommendations_path).select(pl.len()).collect().item()
if saved_rows != expected_rows:
    raise ValueError(f'Expected {expected_rows:,} rows, found {saved_rows:,}')

print(f'Saved candidate rows: {saved_rows:,}')
print(pl.read_parquet(recommendations_path).head())

Saved candidate rows: 167,456,500
shape: (5, 3)
┌────────────────┬────────┬───────────┐
│ impression_uid ┆ cutoff ┆ news_code │
│ ---            ┆ ---    ┆ ---       │
│ i64            ┆ i16    ┆ i32       │
╞════════════════╪════════╪═══════════╡
│ 10996690       ┆ 1      ┆ 95525     │
│ 10996690       ┆ 2      ┆ 53474     │
│ 10996690       ┆ 3      ┆ 13761     │
│ 10996690       ┆ 4      ┆ 98178     │
│ 10996690       ┆ 5      ┆ 20990     │
└────────────────┴────────┴───────────┘


## 5. Save the manifest

In [ ]:
manifest = {
    'model': 'popularity',
    'score': 'prior_clicks',
    'tie_break': 'news_code ascending',
    'fit_split': FIT_SPLIT,
    'target_split': TARGET_SPLIT,
    'candidate_cap': CANDIDATE_CAP,
    'candidate_format': ['impression_uid', 'cutoff', 'news_code'],
    'eligibility': 'available_from <= timestamp',
    'catalogue_rows': popularity.height,
    'target_impressions': impressions.height,
    'unique_timestamps': n_timestamps,
    'recommendation_rows': expected_rows,
    'model_file': model_path.name,
    'recommendations_file': recommendations_path.name,
    'created_at': datetime.now(timezone.utc).isoformat(),
}
manifest_path = OUTPUT_DIR / f'manifest_{TARGET_SPLIT}.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')

print('Manifest saved to:', manifest_path)
manifest

Manifest saved to: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\popularity\manifest_validation.json


{'model': 'popularity',
 'score': 'prior_clicks',
 'tie_break': 'news_code ascending',
 'fit_split': 'train',
 'target_split': 'validation',
 'candidate_cap': 500,
 'candidate_format': ['impression_uid', 'cutoff', 'news_code'],
 'eligibility': 'available_from <= timestamp',
 'catalogue_rows': 104151,
 'target_impressions': 334913,
 'unique_timestamps': 54728,
 'recommendation_rows': 167456500,
 'model_file': 'model_train.parquet',
 'recommendations_file': 'recommendations_validation_top500.parquet',
 'created_at': '2026-10-22T16:30:34.484151+00:00'}

## 6. Evaluate Recall@M

Expected pooled recall is approximately 0.0406, 0.0925, 0.1367 and 0.2109 for M = 50, 100, 200 and 500.

In [ ]:
evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
evaluation = evaluator.evaluate(
    recommendations_path,
    model_name='popularity',
    m_values=M_VALUES,
)

evaluation_path = ROOT / 'results' / f'candidate_generation_popularity_{TARGET_SPLIT}.csv'
evaluation.write_csv(evaluation_path)
print('Evaluation saved to:', evaluation_path)
evaluation

Evaluation saved to: c:\Users\Yong Rui\BT4222-Group-6-\results\candidate_generation_popularity_validation.csv


model,split,m,recall_pooled,recall_impression_mean,hit_rate,impressions,relevant_clicks,retrieved_relevant,mean_candidates,mean_eligible_pool,eligible_click_coverage
str,str,i64,f64,f64,f64,i64,i64,i64,f64,f64,f64
"""popularity""","""validation""",50,0.0028,0.002252,0.004258,334913,514658,1441,50.0,99816.190673,1.0
"""popularity""","""validation""",100,0.013557,0.012005,0.020337,334913,514658,6977,100.0,99816.190673,1.0
"""popularity""","""validation""",200,0.046281,0.044201,0.067412,334913,514658,23819,200.0,99816.190673,1.0
"""popularity""","""validation""",500,0.178921,0.186907,0.240877,334913,514658,92083,500.0,99816.190673,1.0


## 7. What this baseline does and does not do

- It is non-personalised: every user receives the same popularity order apart from eligibility.
- Articles never clicked in the fit split receive score 0. Roughly 14.4% of validation clicks are on such articles, so this baseline cannot retrieve them.
- It is frozen per split: popularity is not updated within the target window. This is intentional for the first baseline.
- `available_from` is an observed-availability proxy, not a verified publication timestamp. Eligibility follows the project-wide convention.
- The output is standard impression-level candidates, ready for the re-ranker and any candidate-generation evaluator.